# Vocabulary Extension Strategy and Documentation

Given a llama 3 base tokenizer pretrained mainly on high resource langauges we want to extned the vocabulary of the tokeniezr so taht its appropriate for our huge multilingual corpus

However we dont want to extend the vocab in every langauge. E.g we dont need to add english vocabs since the tokenizer is already wenn suited for english data.

Therefore we first identify which languges we need to extend the tokenizer for.
### Step 1: Detect New (Unknown) Languages
We test how well the base tokenizer handles well the base tokenizer handles all the languages.

We run the tokenizer on a sample of each language and collect:
- **Characters per token**
- **Pieces per word**
- **Unknown token rate**

Languages showing high fragmentation or many unknown tokens are considered *new or poorly covered* and become candidates for vocabulary extension.
For this step the script compute_tokenizer_coverage.py was written, it extracts the output in .metric/tokenizer_metric.csvimport pandas as pd




In [14]:
import pandas as pd
df = pd.read_csv("/upb/users/j/joeldag/profiles/unix/cs/MERLIN/moe-study/data_prep/metrics/tokenizer_metric_llama3.2-1b.csv")
print("Best covered languages:")
print(df.nsmallest(15, "pieces_per_word")[["language", "pieces_per_word", "chars_per_token", "unknown_rate"]])

print("\n Worst covered languages:")
print(df.nlargest(5, "pieces_per_word")[["language", "pieces_per_word", "chars_per_token", "unknown_rate"]])

Best covered languages:
     language  pieces_per_word  chars_per_token  unknown_rate
36    english         1.312613         4.515410           0.0
152  spa_Latn         1.623986         3.739960           0.0
56   fas_Arab         1.658725         3.037348           0.0
129  nin_Latn         1.709627         2.517444           0.0
61   ile_Latn         1.715052         3.421946           0.0
91   lbb_Latn         1.782157         2.692541           0.0
27   csy_Latn         1.800657         2.725445           0.0
84   hin_Latn         1.815577         2.882508           0.0
170  xmm_Latn         1.864788         2.982819           0.0
81   jbo_Latn         1.881246         2.598243           0.0
139  nii_Latn         1.919537         2.617205           0.0
160  whg_Latn         1.999576         2.816405           0.0
57   ibg_Latn         1.999610         3.037050           0.0
112  mww_Latn         2.005040         2.475768           0.0
104  lgl_Latn         2.008873         2.47812

The unknown token analysis failed, since llama3 tokenizer doesnt have a unknown token:

In [15]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B", use_fast=True)
print("unk_token:", tokenizer.unk_token)
print("unk_token_id:", tokenizer.unk_token_id)


unk_token: None
unk_token_id: None


I also tried to include multilingual evaluation benchmarks such as Belebele and HellaSwag. However, I decided not to, since many languages were not covered, the evaluation process was very time-consuming, and the language mapping (e.g., eng_Latn → English) between datasets required significant manual work. Additionally, the LM evaluation framework didn’t support some of the languages I wanted — for example, Belebele only provided 31 languages instead of 122.

### Extend of vocabulary extension for each language?
We aim to expand the LLaMA-3 tokenizer from its original 128K vocabulary to 250K tokens, resulting in an additional 122K subword units.
Rather than distributing these new tokens uniformly, we allocate them proportionally to each language’s tokenizer performance.
Languages that exhibit higher fragmentation or lower coverage (i.e., where the base tokenizer performs poorly) receive a larger share of new tokens, while well-covered languages receive fewer or none.
This adaptive allocation ensures that the extended vocabulary grows efficiently and improves coverage where it is most needed.

### Vocabulary extension: ranking-based, no thresholds (concise walkthrough)
We expand LLaMA-3 from **128K → 250K** (+**122K** tokens).
Each language gets tokens **proportional to a continuous “inefficiency” score** derived from:
- higher = worse: *pieces per word* (PPW)
- lower = worse: *characters per token* (CPT)
- (optional) *unknown rate* (UNK; here mostly 0 so weight = 0)

We emphasize under-covered languages by raising the score to a power γ>1.

In [16]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

TOTAL_NEW_TOKENS = 122_000
W_PPW, W_CPT, W_UNK = 0.5, 0.5, 0.0   # weights for each metric, Note: unknown token id does not exist for out tokenizer
GAMMA = 2.0                            # nonlinearity (γ > 1 emphasizes bad langs)
CSV_PATH = "/upb/users/j/joeldag/profiles/unix/cs/MERLIN/moe-study/data_prep/metrics/tokenizer_metric_llama3.2-1b.csv"
df = pd.read_csv(CSV_PATH)
df.head()


Loading tokenizer metrics...
Loaded 200 languages.



,language,chars_per_token,pieces_per_word,unknown_rate,num_texts
0,bwq_Latn,1.881597,2.816398,0.0,5000
1,bon_Latn,2.505693,2.526153,0.0,5000
2,cho_Latn,2.194035,2.981325,0.0,5000
3,ayo_Latn,2.480992,2.273810,0.0,5000
4,bgs_Latn,2.522694,2.207313,0.0,5000


In [17]:
# normalize metrics
scaler = MinMaxScaler()
dfn = df.copy()
dfn["ppw_norm"] = scaler.fit_transform(dfn[["pieces_per_word"]])
dfn["cpt_norm"] = 1 - scaler.fit_transform(dfn[["chars_per_token"]])
dfn["unk_norm"] = scaler.fit_transform(dfn[["unknown_rate"]])
dfn[["language", "ppw_norm", "cpt_norm", "unk_norm"]].head()

Metrics normalized (PPW↑, CPT↓, UNK↑).


,language,ppw_norm,cpt_norm,unk_norm
0,bwq_Latn,0.039394,0.642515,0.0
1,bon_Latn,0.031791,0.490268,0.0
2,cho_Latn,0.043715,0.566297,0.0
3,ayo_Latn,0.025180,0.496294,0.0
4,bgs_Latn,0.023438,0.486121,0.0


In [18]:
# linear inefficiency score that indicated how inefficient the tokenizer is in a specific language
dfn["I_linear"] = (
    W_PPW * dfn["ppw_norm"] +
    W_CPT * dfn["cpt_norm"] +
    W_UNK * dfn["unk_norm"]
)

print("linear inefficiency scores: ")
dfn[["language", "I_linear"]].head()


Computed linear inefficiency score.


,language,I_linear
0,bwq_Latn,0.340955
1,bon_Latn,0.261029
2,cho_Latn,0.305006
3,ayo_Latn,0.260737
4,bgs_Latn,0.254779


In [19]:
# Apply nonlinear scaling to emphasize under-covered languages
dfn["I_nonlinear"] = np.power(dfn["I_linear"], GAMMA)
dfn["inefficiency_score"] = MinMaxScaler().fit_transform(dfn[["I_nonlinear"]])

print(f"Applied nonlinear scaling (gamma = {GAMMA}).")
dfn = dfn.sort_values("inefficiency_score", ascending=False)
dfn[["language", "pieces_per_word", "chars_per_token", "inefficiency_score"]].head(10)


Applied nonlinear scaling (gamma = 2.0).


,language,pieces_per_word,chars_per_token,inefficiency_score
63,dzo_Tibt,39.485451,0.532245,1.000000
115,ksw_Mymr,33.136879,0.512866,0.842812
93,lao_Laoo,28.098671,0.563538,0.713753
164,wuu_Hani,23.733892,1.200818,0.501262
137,ory_Orya,15.522427,0.416189,0.484384
90,kat_Geor,14.866699,0.556227,0.448817
186,ojb_Cans,13.620305,0.423909,0.448564
159,sat_Olck,13.084546,0.435631,0.437160
172,tzm_Tfng,12.183218,0.456195,0.418170
146,tig_Ethi,11.320000,0.425206,0.408354


In [20]:
#  now we allocate total new tokens proportionally to inefficiency scores
weights = dfn["inefficiency_score"].to_numpy()
total_w = weights.sum()

if total_w == 0:
    dfn["token_alloc"] = 0
else:
    ideal = weights / total_w * TOTAL_NEW_TOKENS
    floor = np.floor(ideal).astype(int)
    remainder = ideal - floor
    shortfall = TOTAL_NEW_TOKENS - floor.sum()
    if shortfall > 0:
        idx = np.argsort(-remainder)[:shortfall]
        floor[idx] += 1
    dfn["token_alloc"] = floor

assert dfn["token_alloc"].sum() == TOTAL_NEW_TOKENS, "Check allocation sum!"
print(f"Sum of allocated tokens: {dfn['token_alloc'].sum()} (target = {TOTAL_NEW_TOKENS})")


Sum of allocated tokens: 122000 (target = 122000)


In [21]:
print("\nbest 15 languages with token allocation:")
print(dfn[["language", "inefficiency_score", "token_alloc"]].head(15))

print("\nworst 10 (best-covered) languages:")
print(dfn[["language", "inefficiency_score", "token_alloc"]].tail(10))



Top 15 languages by token allocation:
     language  inefficiency_score  token_alloc
63   dzo_Tibt            1.000000         4585
115  ksw_Mymr            0.842812         3864
93   lao_Laoo            0.713753         3272
164  wuu_Hani            0.501262         2298
137  ory_Orya            0.484384         2221
90   kat_Geor            0.448817         2058
186  ojb_Cans            0.448564         2057
159  sat_Olck            0.437160         2004
172  tzm_Tfng            0.418170         1917
146  tig_Ethi            0.408354         1872
194  tel_Telu            0.407948         1870
47   crl_Cans            0.400360         1836
179  tcy_Knda            0.399048         1830
199  syc_Syrc            0.389199         1784
178  wsg_Telu            0.378428         1735

Bottom 10 (best-covered) languages:
     language  inefficiency_score  token_alloc
131  pam_Latn            0.042489          195
170  xmm_Latn            0.038792          178
57   ibg_Latn            0.0368